In [ ]:
import os, glob

import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from bluemath_tk.datamining import PCA
from sklearn.decomposition import PCA as PCA_
from bluemath_tk.core.operations import spatial_gradient
from joblib import Parallel, delayed

#### Load Data

In [ ]:
pcs_df = pd.read_csv('outputs/pca/pcs_msl_30_6h.csv', index_col=0, parse_dates=True)
cols = list(pcs_df.columns)

pca_msl = PCA.load_model('outputs/pca/pca_msl.pkl')

era5 = xr.open_zarr('data/era5_mslp_grad_1985_2026_1deg_hourly.zarr').rename({'valid_time': 'time'})
era5 = era5.sel(time=era5.time.dt.hour.isin([0, 6, 12, 18]))

training = slice('1985-01-01', '2015-12-31')

sim_files = sorted(glob.glob('outputs/var_6h/simulaciones_h1/sim_*.parquet'))
print(len(sim_files))

#### PCA del gradiente real (bluemath)

In [ ]:
pca_grad = PCA(n_components=50)
pca_grad._pca = PCA_(n_components=50, svd_solver='covariance_eigh')
pca_grad.fit_transform(data=era5[['grad2']].sel(time=training), vars_to_stack=['grad2'],
                       coords_to_stack=['latitude', 'longitude'], pca_dim_for_rows='time')

cum = np.cumsum(pca_grad.explained_variance_ratio)
print(f'10 PCs: {cum[9]:.3f} | 50 PCs: {cum[49]:.3f}')

#### PCs de presión → campo (inverse_transform) → gradiente → PCs del gradiente

In [ ]:
def grad_pcs(pcs, n_gr=10, paso=8000):
    out = []
    for i in range(0, len(pcs), paso):
        p = pcs.iloc[i:i + paso][cols]
        pcs_da = xr.DataArray(p.values, dims=('time', 'n_component'),
                              coords={'time': p.index, 'n_component': np.arange(len(cols))})
        campo = pca_msl.inverse_transform(pcs_da)['msl'].transpose('time', 'latitude', 'longitude')
        grad = spatial_gradient(campo).to_dataset(name='grad2')
        out.append(pca_grad.transform(data=grad)['PCs'].values[:, :n_gr])
    return pd.DataFrame(np.concatenate(out), index=pcs.index, columns=[f'grec_PC{k + 1}' for k in range(n_gr)])

#### Histórico

In [ ]:
grec_hist = grad_pcs(pcs_df)

#### Comprobación: reconstruido vs real en training
grec_real = pca_grad.transform(data=era5[['grad2']].sel(time=training))['PCs'].values[:, :10]
t_train = grec_hist.loc[training].index
print(pd.Series([np.corrcoef(grec_real[:, k], grec_hist.loc[t_train].values[:, k])[0, 1] for k in range(10)],
                index=grec_hist.columns).round(2))

In [ ]:
os.makedirs('outputs/grad_rec', exist_ok=True)
grec_hist.to_csv('outputs/grad_rec/pcs_grad_rec_6h.csv', float_format='%.5f')
pca_grad.save_model('outputs/grad_rec/pca_grad.pkl')

#### Simulaciones h1

In [ ]:
out_dir = 'outputs/grad_rec/simulaciones_h1'
os.makedirs(out_dir, exist_ok=True)

def procesar(f):
    f_out = os.path.join(out_dir, os.path.basename(f).replace('sim_', 'grec_'))
    if not os.path.exists(f_out):
        grad_pcs(pd.read_parquet(f)).astype('float32').to_parquet(f_out)

Parallel(n_jobs=os.cpu_count() - 2, verbose=5)(delayed(procesar)(f) for f in sim_files)